### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="iranian_churn",
    dataset_year="2011",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5JW3Z",
    download_description="""
mkdir -p local-data-warehouse/iranian_churn \
&& wget -P local-data-warehouse/iranian_churn/ https://archive.ics.uci.edu/static/public/563/iranian+churn+dataset.zip \
&& unzip local-data-warehouse/iranian_churn/iranian+churn+dataset.zip -d local-data-warehouse/iranian_churn/ \
&& rm local-data-warehouse/iranian_churn/iranian+churn+dataset.zip
""",
    # References
    academic_reference_bibtex="""@article{keramati2011churn,
  title={Churn analysis for an Iranian mobile operator},
  author={Keramati, Abbas and Ardabili, Seyed MS},
  journal={Telecommunications Policy},
  volume={35},
  number={4},
  pages={344--356},
  year={2011},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="keramati2011churn",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- The data collection process for the task is not ideal. The data was collected for each individual right up to the churn month. That means, for the churners we observed varying length periods, while for the other individuals we observed the full period.
- Nevertheless, the features are not collected after the churn and the task is still valid, if we conceptualize it as “identify customers close to churn” rather than “predict churn ahead of time”.
- We remove exact duplicates (9.52% of the samples) from the data to avoid leaks. The data contains features like "Seconds of Use", "Frequency of use", "Frequency of SMS", "Distinct Called Numbers", and most importantly a customer value assignment. Given just 3000 samples, it is reasonable to assume that if feature values with so many degrees of freedom are exactly the same, they belong to the same customer.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Churn",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Churn",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "Customer Churn.csv")
df = df.drop_duplicates().reset_index(drop=True)

df["Tariff Plan"] = df["Tariff Plan"].map({1: "Pay as you go", 2: "Contractual"})
df["Status"] = df["Status"].map({1: "Active", 2: "Non-active"})
df["Churn"] = df["Churn"].map({1: "Churn", 0: "Non-churn"})
df["Complains"] = df["Complains"].map({0: "No complaint", 1: "Complaint"})

as_cat_dtype = [
    "Tariff Plan",
    "Complains",
    "Status",
    "Churn",
    # Ordinals which we keep as num
    # "Charge Amount",
    # "Age Group",
]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Loaded data shape:", df.shape)

Loaded data shape: (2850, 14)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 2,850
Columns: 14
Use sampling: False (sample size: 2,850)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Customer Value', 'Seconds of Use', 'Frequency of SMS', 'Frequency of use', 'Distinct Called Numbers', 'Subscription  Length', 'Call  Failure', 'Charge  Amount', 'Age Group', 'Age']
Rows remaining as candidates after top-10 filter: 108 (of 2,850)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 14 (0.49% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,20,No complaint,28,1,15655,226,0,74,2,Pay as you go,Active,25,714.645,Non-churn
1,8,Complaint,30,0,5558,88,11,18,2,Pay as you go,Active,25,303.570,Churn
2,14,No complaint,22,5,3238,53,48,25,3,Contractual,Active,30,323.640,Non-churn
3,2,No complaint,35,0,2073,30,1,4,3,Pay as you go,Active,30,88.120,Non-churn
4,12,No complaint,44,2,4283,98,24,36,3,Pay as you go,Active,30,271.240,Non-churn


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Complains,category,0.0,0.0,2.0,"No complaint, Complaint"
1,Tariff Plan,category,0.0,0.0,2.0,"Pay as you go, Contractual"
2,Status,category,0.0,0.0,2.0,"Active, Non-active"
3,Churn,category,0.0,0.0,2.0,"Non-churn, Churn"
4,Customer Value,float64,0.0,0.0,2654.0,"0.0, 40.44, 45.495, 15.165, 25.275, 159.42, 131.4, 734.085, 383.22, 168.075"
5,Call Failure,int64,0.0,0.0,37.0,"0, 5, 7, 6, 8, 9, 3, 2, 4, 10"
6,Subscription Length,int64,0.0,0.0,45.0,"36, 38, 37, 35, 39, 34, 40, 33, 32, 41"
7,Charge Amount,int64,0.0,0.0,11.0,"0, 1, 2, 3, 4, 5, 8, 9, 7, 6"
8,Seconds of Use,int64,0.0,0.0,1756.0,"0, 305, 2088, 955, 1015, 710, 1973, 825, 650, 2475"
9,Frequency of use,int64,0.0,0.0,242.0,"0, 6, 44, 41, 39, 36, 35, 30, 50, 42"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Call Failure,2850.0,7.802456,7.326172,0.0,36.00
Subscription Length,2850.0,32.452982,8.723075,3.0,47.00
Charge Amount,2850.0,0.974737,1.550618,0.0,10.00
Seconds of Use,2850.0,4534.243158,4199.712303,0.0,17090.00
Frequency of use,2850.0,70.484912,57.401512,0.0,255.00
Frequency of SMS,2850.0,73.789825,112.062397,0.0,522.00
Distinct Called Numbers,2850.0,23.870526,17.193929,0.0,97.00
Age Group,2850.0,2.835088,0.893503,1.0,5.00
Age,2850.0,31.077193,8.861934,15.0,55.00
Customer Value,2850.0,474.990367,514.442198,0.0,2165.28


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column      rank                             
Churn       1         Non-churn   2404  84.35
            2             Churn    446  15.65
Complains   1      No complaint   2620  91.93
            2         Complaint    230   8.07
Status      1            Active   2166  76.00
            2        Non-active    684  24.00
Tariff Plan 1     Pay as you go   2621  91.96
            2       Contractual    229   8.04

In [8]:
# Target Distribution
target_df

,count,pct
Churn,,
Non-churn,2404,84.35
Churn,446,15.65


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to iranian_churn/019d7374-7b3c-7eed-940f-e8ac3c68e2f4
019d7374-7b3c-7eed-940f-e8ac3c68e2f4
97f8db4094085f4777a7cc8fdac237ac8fb6ef7147c25a6a5443d489b5ad27c4
